# Layer Communication

Analyze how layers communicate through the residual stream: message norms,
channel utilization, bandwidth constraints, bypass detection, and inter-layer alignment.

## Why This Matters

Layer communication characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_communication import (
    layer_message_norms,
    channel_utilization,
    bandwidth_analysis,
    layer_bypass_detection,
    inter_layer_alignment,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Layer Message Norms

Measure the magnitude of messages (attention and MLP outputs) that each layer
writes to the residual stream, compared to the residual stream norm.

In [ ]:
result = layer_message_norms(model, tokens)
print("Attention message norms per layer:")
for i, norm in enumerate(result["attn_message_norms"]):
    print(f"  Layer {i}: {norm:.4f}")
print("\nMLP message norms per layer:")
for i, norm in enumerate(result["mlp_message_norms"]):
    print(f"  Layer {i}: {norm:.4f}")
print(f"\nResidual norms: {result['residual_norms']}")
print(f"Message-to-residual ratio: {result['message_to_residual_ratio']}")

## Channel Utilization

Analyze how many dimensions of the residual stream are actively used at each layer.
Low utilization means the model concentrates information in few dimensions.

In [ ]:
result = channel_utilization(model, tokens)
print("Effective dimensions per layer:")
for i, dims in enumerate(result["effective_dims"]):
    print(f"  Layer {i}: {dims:.1f} / {cfg.d_model}")
print(f"\nUtilization per layer: {result['utilization']}")
print(f"Top-dim fraction: {result['top_dim_fraction']}")

## Bandwidth Analysis

Measure information bandwidth at each layer by analyzing the rank/effective
dimensionality of the residual stream deltas (changes per layer).

In [ ]:
result = bandwidth_analysis(model, tokens)
print(f"Layer deltas shape: {result['layer_deltas'].shape}")
print(f"Bandwidth per layer: {result['bandwidth']}")
print(f"Delta ranks: {result['delta_ranks']}")
print(f"Bottleneck layer: {result['bottleneck_layer']}")

## Layer Bypass Detection

Detect layers that can be skipped with minimal impact on the model's output.
Uses skip-connection similarity and metric impact to identify bypassed layers.

In [ ]:
result = layer_bypass_detection(model, tokens, metric_fn)
print("Skip similarity (input vs output per layer):")
for i, sim in enumerate(result["skip_similarity"]):
    print(f"  Layer {i}: {sim:.4f}")
print(f"\nMetric impact per layer: {result['metric_impact']}")
print(f"Is bypass: {result['is_bypass']}")
print(f"Number of bypass layers: {result['n_bypass_layers']}")

## Inter-Layer Alignment

Measure how aligned the outputs of attention and MLP are within each layer,
and how aligned consecutive layers' residual stream updates are.

In [ ]:
result = inter_layer_alignment(model, tokens)
print("Cosine alignment (consecutive layer residuals):")
for i, align in enumerate(result["cosine_alignment"]):
    print(f"  Layer {i}: {align:.4f}")
print(f"\nAttn-MLP alignment per layer: {result['attn_mlp_alignment']}")
print(f"Mean alignment: {result['mean_alignment']:.4f}")